# Importing Packages

python scripts/reformat_hlagyn.py --datadir data --rename data/rename_columns.xlsx --correction data/fix_values.xlsx --output combined_test.tsv

In [1]:
import pandas as pd
import os
import re

## Loading Data

In [2]:
def load_table(file, separator=None):
    df = ''
    if str(file).split('.')[-1] == 'tsv':
        separator = '\t' if separator is None else separator
        df = pd.read_csv(file, encoding='latin-1', sep=separator, dtype='str')
    elif str(file).split('.')[-1] == 'csv':
        separator = ',' if separator is None else separator
        df = pd.read_csv(file, encoding='latin-1', sep=separator, dtype='str')
    elif str(file).split('.')[-1] in ['xls', 'xlsx']:
        df = pd.read_excel(file, index_col=None, header=0, sheet_name=0, dtype='str')
        df.fillna('', inplace=True)
    else:
        print('Wrong file format. Compatible file formats: TSV, CSV, XLS, XLSX')
        exit()
    return df

In [3]:
BASE_PATH = '../data/HLAGyn/'
dfs = []

dfs_para_influ = []
dfs_ph4 = []
dfs_covid = []
dfts_ctn = []

for file in os.listdir(BASE_PATH):
    if file.endswith(('.xls', '.xlsx')):
        print(file)
        df = load_table(BASE_PATH + file).assign(filename=file)
        dfs.append(df)

        columns = ''.join(df.columns.tolist())
        h1n1_column = 'H1N1' in columns or 'Influenza' in columns
        ctn_column = 'CT_N' in columns

        if h1n1_column:
            if 'Parainfluenza' in columns:
                dfs_para_influ.append(df)
            elif 'PH4' in columns:
                dfs_ph4.append(df)
            elif 'SARS-CoV-2' in columns:
                dfs_covid.append(df)
        elif ctn_column:
            dfts_ctn.append(df)
        else:
            print('Unknown type')

20220329_Painel Viral 24_Segunda quinzena de Março de 2022.xlsx
20220912_PR24_SE36.xlsx
20230130_COVID_SE04.xlsx
20221121_PR24_SE46.xlsx
20230320_PR4_SE11.xlsx
20230102_PR4_SE52.xlsx
20230116_PR4_SE02.xlsx
20221210_COVID_SE49.xlsx
20220926_PR4_SE38.xlsx
20230605_PR24_SE22.xlsx
20230808_PR24_SE31.xlsx
20220614_PR4_se23.xlsx
20220627_PR4_SE25.xlsx
20220912_Covid_SE36.xlsx
20230320_COVID_SE11.xlsx
20230130_PR4_SE04.xlsx
20230731_PR24_SE30.xlsx
20220823_PR4_SE33.xlsx
20230626_PR4_SE25.xlsx
20220601_PR4_se21.xlsx
20220808_PR24_SE31.xlsx
20220510_PR24_se18.xlsx
20230313_PR4_SE10.xlsx
20230403_PR4_SE13.xlsx
20230130_PR24_SE04.xlsx
20220912_PR4_SE36.xlsx
20230208_PR24_SE05.xlsx
20230717_PR4_SE28.xlsx
20220504_Painel Viral 24_SE 17_2022.xlsx
20230102_COVID_SE52.xlsx
20230529_PR24_SE21.xlsx
20230701_PR24_SE26.xlsx
20230213_PR4_SE06.xlsx
20230307_PR24_SE09.2023.xlsx
20230307_PR4_SE09.2023.xlsx
20220823_PR24_SE33.xlsx
20221121_COVID_SE46.xlsx
20220704_PR4_SE26.xlsx
20220919_PR24_SE37.xlsx
20221205

In [4]:
df_hla_flu = pd.concat(dfs_para_influ)
df_hla_ph4 = pd.concat(dfs_ph4)
df_hla_covid = pd.concat(dfs_covid)
df_hla_ctn = pd.concat(dfts_ctn)

## EDA

### PH4

In [5]:
print(df_hla_ph4.shape)
df_hla_ph4.info()

(1636, 39)
<class 'pandas.core.frame.DataFrame'>
Index: 1636 entries, 0 to 34
Data columns (total 39 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Idade           1636 non-null   object
 1   Sexo            1636 non-null   object
 2   Pedido          1636 non-null   object
 3   Data Coleta     1636 non-null   object
 4   VIRUS_IA        1636 non-null   object
 5   VIRUS_H1N1      1636 non-null   object
 6   VIRUS_AH3       1636 non-null   object
 7   VIRUS_B         1636 non-null   object
 8   VIRUS_MH        1636 non-null   object
 9   VIRUS_SA        1636 non-null   object
 10  VIRUS_SB        1636 non-null   object
 11  VIRUS_RH        1636 non-null   object
 12  VIRUS_PH        1636 non-null   object
 13  VIRUS_PH2       1636 non-null   object
 14  VIRUS_PH3       1636 non-null   object
 15  VIRUS_PH4       1636 non-null   object
 16  VIRUS_ADE       1636 non-null   object
 17  VIRUS_BOC       1636 non-null   object
 18  VIRU

In [6]:
df_hla_ph4.columns

Index(['Idade', 'Sexo', 'Pedido', 'Data Coleta', 'VIRUS_IA', 'VIRUS_H1N1',
       'VIRUS_AH3', 'VIRUS_B', 'VIRUS_MH', 'VIRUS_SA', 'VIRUS_SB', 'VIRUS_RH',
       'VIRUS_PH', 'VIRUS_PH2', 'VIRUS_PH3', 'VIRUS_PH4', 'VIRUS_ADE',
       'VIRUS_BOC', 'VIRUS_229E', 'VIRUS_HKU', 'VIRUS_NL63', 'VIRUS_OC43',
       'VIRUS_SARS', 'VIRUS_COV2', 'VIRUS_EV', 'BACTE_BP', 'BACTE_BPAR',
       'BACTE_MP', 'Tipo Material', 'Dt Liberação', 'Método', 'Cidade', 'UF',
       'filename', 'Metodologia', 'Métodologia', 'Metodo', 'Dt. Nascimento',
       'METODOLOGIA'],
      dtype='object')

In [7]:
df_hla_ph4.head()

,Idade,Sexo,Pedido,Data Coleta,VIRUS_IA,VIRUS_H1N1,VIRUS_AH3,VIRUS_B,VIRUS_MH,VIRUS_SA,...,Dt Liberação,Método,Cidade,UF,filename,Metodologia,Métodologia,Metodo,Dt. Nascimento,METODOLOGIA
0,10,Feminino,2284803,2022-09-04 00:00:00,Detectado,Não Detectado,Detectado,Não Detectado,Não Detectado,Não Detectado,...,2022-09-04 00:00:00,FLOW CHIP,APARECIDA DE GOIANIA,GO,20220912_PR24_SE36.xlsx,NaN,NaN,NaN,NaN,NaN
1,2,Masculino,2284815,2022-09-03 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Não Detectado,...,2022-09-04 00:00:00,FLOW CHIP,APARECIDA DE GOIANIA,GO,20220912_PR24_SE36.xlsx,NaN,NaN,NaN,NaN,NaN
2,87,Masculino,2284819,2022-09-04 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Não Detectado,...,2022-09-04 00:00:00,FLOW CHIP,GOIANIA,GO,20220912_PR24_SE36.xlsx,NaN,NaN,NaN,NaN,NaN
3,3,Feminino,2284849,2022-09-04 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Não Detectado,...,2022-09-05 00:00:00,FLOW CHIP,RIO VERDE,GO,20220912_PR24_SE36.xlsx,NaN,NaN,NaN,NaN,NaN
4,0,Masculino,2284991,2022-09-05 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Não Detectado,...,2022-09-05 00:00:00,FLOW CHIP,GOIANIA,GO,20220912_PR24_SE36.xlsx,NaN,NaN,NaN,NaN,NaN


In [8]:
# (
#     df_hla_ph4
#     .assign(duplicated=df_hla_ph4.duplicated(subset=["Pedido"]))
#     .query("duplicated == True")
#     ['Pedido'].tolist()
# )

In [9]:
df_hla_ph4['Idade'].unique()

array(['10', '2', '87', '3', '0', '26', '14', '81', '71', '69', '83',
       '21', '37', '68', '1', '8', '5', '84', '34', '31', '70', '4', '80',
       '', '82', '85', '75', '76', '62', '17', '9', '78', '64', '33',
       '63', '35', '61', '29', '32', '53', '49', '15', '25', '11', '79',
       '39', '6', '44', '86', '72', '54', '38', '58', '73', '94', '41',
       '67', '88', '28', '45', '65', '23', '30', '91', '20', '36', '43',
       '55', '90', '60', '16', '40', '92', '13', '93', '66', '52', '42',
       '7', '51', '96', '59', '22', '12', '74', '56', '27', '50', '77',
       '46', '47', '24', '89', '57', '19', '98', '18', '95', '99', '48',
       '97', '65535', '102'], dtype=object)

In [39]:
df_hla_ph4.query("Idade == '65535'")

,Idade,Sexo,Pedido,Data Coleta,VIRUS_IA,VIRUS_H1N1,VIRUS_AH3,VIRUS_B,VIRUS_MH,VIRUS_SA,...,Dt Liberação,Método,Cidade,UF,filename,Metodologia,Métodologia,Metodo,Dt. Nascimento,METODOLOGIA
4,65535,,2265600,2022-07-11 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Não Detectado,...,2022-07-14 00:00:00,Flow Chip,RIO DE JANEIRO,RJ,20220727_PR24_SE28.xlsx,NaN,NaN,NaN,NaN,NaN
5,65535,,2265601,2022-07-11 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Não Detectado,...,2022-07-13 00:00:00,Flow Chip,RIO DE JANEIRO,RJ,20220727_PR24_SE28.xlsx,NaN,NaN,NaN,NaN,NaN
6,65535,,2265602,2022-07-11 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Não Detectado,...,2022-07-13 00:00:00,Flow Chip,RIO DE JANEIRO,RJ,20220727_PR24_SE28.xlsx,NaN,NaN,NaN,NaN,NaN


In [10]:
df_hla_ph4['Sexo'].unique()

array(['Feminino', 'Masculino', '', 'MN'], dtype=object)

In [11]:
df_hla_ph4['Data Coleta'].replace(re.compile('[0-9]'), 'x').unique()

array(['xxxx-xx-xx xx:xx:xx'], dtype=object)

In [12]:
df_hla_ph4['Dt Liberação'].replace(re.compile('[0-9]'), 'x').unique()

array(['xxxx-xx-xx xx:xx:xx', nan], dtype=object)

In [13]:
virus_columns = [ col for col in df_hla_ph4.columns if col.startswith('VIRUS') ]
unique_values = set()
for col in virus_columns:
    unique_values = unique_values.union(set(df_hla_ph4[col].unique()))

unique_values

{'Detectado', 'Não Detectado'}

- Drop duplicated Pedido
- Filter columns

        ``
        PH4_COLUMNS = [
            'Pedido', 
            'Idade', 'Dt. Nascimento',
            
            'Sexo', 
            'Data Coleta', 'Dt Liberação', 
            'Cidade', 'UF',

            'VIRUS_IA', 'VIRUS_H1N1',
            'VIRUS_AH3', 'VIRUS_B', 'VIRUS_MH', 'VIRUS_SA', 'VIRUS_SB', 'VIRUS_RH',
            'VIRUS_PH', 'VIRUS_PH2', 'VIRUS_PH3', 'VIRUS_PH4', 'VIRUS_ADE',
            'VIRUS_BOC', 'VIRUS_229E', 'VIRUS_HKU', 'VIRUS_NL63', 'VIRUS_OC43',
            'VIRUS_SARS', 'VIRUS_COV2', 'VIRUS_EV', 'BACTE_BP', 'BACTE_BPAR',
            'BACTE_MP',
        ]
        ``

- Fix idade -> int e remover >150
- Fix sexo -> F ou M se Masculino ou Feminino e outros -> ''
- Fix Data Coleta -> 'xxxx-xx-xx xx:xx:xx' -> 'yyyy-mm-dd'
- Cidade, UF ok
- Fix VIRUS_* -> Detectado, Não Detectado -> Neg, Pos
- test_kit -> test_24

### Parainfluenza

In [22]:
print(df_hla_flu.shape)
df_hla_flu.info()

(1465, 36)
<class 'pandas.core.frame.DataFrame'>
Index: 1465 entries, 0 to 65
Data columns (total 36 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   Nome Paciente                   1173 non-null   object
 1   Dt. Nascimento                  1447 non-null   object
 2   Idade                           1465 non-null   object
 3   Sexo                            1465 non-null   object
 4   Pedido                          1465 non-null   object
 5   Data Coleta                     1465 non-null   object
 6   VIRUS_Influenza A               1465 non-null   object
 7   VIRUS_Influenza H1N1            1465 non-null   object
 8   VIRUS_Influenza H3              1465 non-null   object
 9   VIRUS_Influenza B               1465 non-null   object
 10  VIRUS_Metapneumovírus           1465 non-null   object
 11  VIRUS_Sincicial A               1465 non-null   object
 12  VIRUS_Sincicial B               1465 non-nul

In [37]:
df_hla_flu.columns

FLU_COLUMNS = [
    'Pedido',
    'Nome Paciente', 'Nome cliente',

    'Idade', 'Dt. Nascimento',
    'Sexo',
    'Cidade', 'UF',


    'Data Coleta', 
    
    'VIRUS_Influenza A', 'VIRUS_Influenza H1N1',
    'VIRUS_Influenza H3', 'VIRUS_Influenza B', 'VIRUS_Metapneumovírus',
    'VIRUS_Sincicial A', 'VIRUS_Sincicial B', 'VIRUS_Rinovírus',
    'VIRUS_Parainfluenza 1', 'VIRUS_Parainfluenza 2',
    'VIRUS_Parainfluenza 3', 'VIRUS_Parainfluenza 4', 'VIRUS_Adenovirus',
    'VIRUS_Bocavirus', 'VIRUS_CoV-229E', 'VIRUS_CoV-HKU', 'VIRUS_CoV-NL63',
    'VIRUS_CoV-OC43', 'VIRUS_SARS_Like', 'VIRUS_SARS-CoV-2',
    'VIRUS_Enterovírus', 'BACTE_Bordetella pertussis',
    'BACTE_Bordetella parapertussis', 'BACTE_Mycoplasma pneumoniae',
]

In [36]:
(
    df_hla_flu
    .assign(duplicated=df_hla_flu.duplicated(subset=["Pedido"]))
    .query("duplicated == True")
    ['Pedido'].tolist()[:5]
)

['2192425', '2192426', '2192497', '2200444', '2200437']

In [30]:
df_hla_flu.query("Pedido in ('2192425','2192426','2192497')").sort_values("Pedido")

,Nome Paciente,Dt. Nascimento,Idade,Sexo,Pedido,Data Coleta,VIRUS_Influenza A,VIRUS_Influenza H1N1,VIRUS_Influenza H3,VIRUS_Influenza B,...,VIRUS_Enterovírus,BACTE_Bordetella pertussis,BACTE_Bordetella parapertussis,BACTE_Mycoplasma pneumoniae,Metodologia Do Teste,Tipo Material,Cidade,UF,filename,Nome cliente
1,EGB,1962-05-15 00:00:00,59,Masculino,2192425,2022-03-12 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,...,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Flow Chip,Swab nasofaringe,APARECIDA DE GOIANIA,GO,20220329_Painel Viral 24_Segunda quinzena de M...,NaN
81,EGB,1962-05-15 00:00:00,59,Masculino,2192425,2022-03-12 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,...,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Flow Chip,Swab nasofaringe,APARECIDA DE GOIANIA,GO,20220314_Painel viral 24_FEV.MAR.2022.xlsx,NaN
2,BPG,1931-12-30 00:00:00,90,Masculino,2192426,2022-03-12 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,...,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Flow Chip,Swab nasofaringe,APARECIDA DE GOIANIA,GO,20220329_Painel Viral 24_Segunda quinzena de M...,NaN
82,BPG,1931-12-30 00:00:00,90,Masculino,2192426,2022-03-12 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,...,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Flow Chip,Swab nasofaringe,APARECIDA DE GOIANIA,GO,20220314_Painel viral 24_FEV.MAR.2022.xlsx,NaN
3,MDFOC,1961-03-14 00:00:00,60,Feminino,2192497,2022-03-12 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,...,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Flow Chip,Swab nasofaringe/orofaringe,APARECIDA DE GOIANIA,GO,20220329_Painel Viral 24_Segunda quinzena de M...,NaN
83,MDFOC,1961-03-14 00:00:00,60,Feminino,2192497,2022-03-12 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,...,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Flow Chip,Swab nasofaringe/orofaringe,APARECIDA DE GOIANIA,GO,20220314_Painel viral 24_FEV.MAR.2022.xlsx,NaN


In [31]:
df_hla_flu['Idade'].unique()

array(['1', '59', '90', '60', '0', '20', '2', '43', '13', '30', '4', '5',
       '61', '29', '46', '7', '23', '58', '53', '63', '22', '9', '3',
       '69', '70', '80', '45', '88', '77', '41', '71', '64', '40', '16',
       '26', '10', '86', '55', '32', '31', '73', '12', '72', '8', '11',
       '82', '48', '36', '6', '47', '52', '74', '83', '65', '24', '28',
       '76', '87', '54', '49', '62', '15', '33', '21', '25', '39', '57',
       '34', '35', '78', '27', '17', '75', '38', '18', '85', '44', '93',
       '56', '79', '67', '92', '19', '96', '14', '51', '37', '42', '50',
       '66', '94', '81', '84', '68', '89', '91', '97', '98', '65535'],
      dtype=object)

In [32]:
df_hla_flu.query("Idade == '65535'")

,Nome Paciente,Dt. Nascimento,Idade,Sexo,Pedido,Data Coleta,VIRUS_Influenza A,VIRUS_Influenza H1N1,VIRUS_Influenza H3,VIRUS_Influenza B,...,VIRUS_Enterovírus,BACTE_Bordetella pertussis,BACTE_Bordetella parapertussis,BACTE_Mycoplasma pneumoniae,Metodologia Do Teste,Tipo Material,Cidade,UF,filename,Nome cliente
894,CPR0,2022-01-24 00:00:00,65535,Masculino,2084605,2022-01-18 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,...,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Flow Chip,Outro material,RIO DE JANEIRO,RJ,20220131_Painel viral_JAN.2022.xlsx,NaN
895,CPR0,2022-01-24 00:00:00,65535,Masculino,2084606,2022-01-18 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,...,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Flow Chip,Outro material,RIO DE JANEIRO,RJ,20220131_Painel viral_JAN.2022.xlsx,NaN
897,CPR0,2022-01-24 00:00:00,65535,Masculino,2084666,2022-01-18 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,...,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Flow Chip,Outro material,RIO DE JANEIRO,RJ,20220131_Painel viral_JAN.2022.xlsx,NaN


In [40]:
df_hla_flu['Sexo'].unique()

array(['Masculino', 'Feminino', 'FEMININO'], dtype=object)

In [42]:
df_hla_flu['Data Coleta'].replace(re.compile('[0-9]'), 'x').unique()

array(['xxxx-xx-xx xx:xx:xx'], dtype=object)

In [44]:
virus_columns = [ col for col in df_hla_flu.columns if col.startswith('VIRUS') or col.startswith('BACTE') ]
unique_values = set()
for col in virus_columns:
    unique_values = unique_values.union(set(df_hla_flu[col].unique()))

unique_values

{'Detectado', 'Não Detectado'}

- Drop duplicated Pedido
- Filter columns

        ``
        PARAFLU_COLUMNS = [
            'Pedido',

            'Idade', 'Dt. Nascimento',
            'Sexo',
            'Cidade', 'UF',


            'Data Coleta', 
            
            'VIRUS_Influenza A', 'VIRUS_Influenza H1N1',
            'VIRUS_Influenza H3', 'VIRUS_Influenza B', 'VIRUS_Metapneumovírus',
            'VIRUS_Sincicial A', 'VIRUS_Sincicial B', 'VIRUS_Rinovírus',
            'VIRUS_Parainfluenza 1', 'VIRUS_Parainfluenza 2',
            'VIRUS_Parainfluenza 3', 'VIRUS_Parainfluenza 4', 'VIRUS_Adenovirus',
            'VIRUS_Bocavirus', 'VIRUS_CoV-229E', 'VIRUS_CoV-HKU', 'VIRUS_CoV-NL63',
            'VIRUS_CoV-OC43', 'VIRUS_SARS_Like', 'VIRUS_SARS-CoV-2',
            'VIRUS_Enterovírus', 'BACTE_Bordetella pertussis',
            'BACTE_Bordetella parapertussis', 'BACTE_Mycoplasma pneumoniae',
        ]
        ``

- Fix idade -> int e remover >150
- Fix sexo -> uppercase -> F ou M se Masculino ou Feminino e outros -> ''
- Fix Data Coleta -> 'xxxx-xx-xx xx:xx:xx' -> 'yyyy-mm-dd'
- Cidade, UF ok
- Fix VIRUS_* -> Detectado, Não Detectado -> Neg, Pos
- test_kit -> test_24

### SARS-CoV-2

In [45]:
df_hla_covid.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1769 entries, 0 to 28
Data columns (total 23 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   Idade                             1769 non-null   object
 1   Sexo                              1769 non-null   object
 2   Pedido                            1769 non-null   object
 3   Data Coleta                       1769 non-null   object
 4   Vírus Influenza A                 1769 non-null   object
 5   Vírus Influenza B                 1769 non-null   object
 6   Vírus Sincicial Respiratório A/B  1769 non-null   object
 7   Coronavírus SARS-CoV-2            1769 non-null   object
 8   Tipo Material                     1769 non-null   object
 9   Dt Liberação                      1334 non-null   object
 10  Método                            588 non-null    object
 11  Cidade                            1769 non-null   object
 12  UF                         

In [47]:
df_hla_covid.columns

COVID_COLUMNS = [
    'Pedido',

    'Idade', 'Sexo',  
    
    'Cidade', 'UF',

    'Data Coleta', 'Dt Liberação',


    'Vírus Influenza A',
    'Vírus Influenza B', 
    'Vírus Sincicial Respiratório A/B',
    'Coronavírus SARS-CoV-2', 
]

In [54]:
(
    df_hla_covid
    .assign(duplicated=df_hla_covid.duplicated(subset=["Pedido"]))
    .query("duplicated == True")
    ['Pedido'].tolist()[:5]
)

['2199883', '2199884', '2199885', '2199886', '2200182']

In [55]:
df_hla_covid.query("Pedido in ['2199883', '2199884', '2199885', '2199886', '2200182']").sort_values("Pedido")

,Idade,Sexo,Pedido,Data Coleta,Vírus Influenza A,Vírus Influenza B,Vírus Sincicial Respiratório A/B,Coronavírus SARS-CoV-2,Tipo Material,Dt Liberação,...,filename,Metodo,Métodologia,Metodologia,Metodologia Do Teste,Nome Paciente,Dt. Nascimento,Cidade.1,Matodologia,Inst Saúde
6,69,Masculino,2199883,2022-03-25 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Swab nasofaringe,NaN,...,20220329_Painel Viral 4_Segunda quinzena de Ma...,NaN,NaN,NaN,RT-qPCR,OPG,1952-09-23 00:00:00,GOIANIA,NaN,NaN
1,69,Masculino,2199883,2022-03-25 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Swab nasofaringe,NaN,...,20220405_Painel Viral 4.xlsx,NaN,NaN,NaN,RT-qPCR,OPG,1952-09-23 00:00:00,NaN,NaN,NaN
7,0,Feminino,2199884,2022-03-25 00:00:00,Não Detectado,Não Detectado,Detectado,Não Detectado,Swab nasofaringe,NaN,...,20220329_Painel Viral 4_Segunda quinzena de Ma...,NaN,NaN,NaN,RT-qPCR,MJCDM,2022-03-05 00:00:00,GOIANIA,NaN,NaN
2,0,Feminino,2199884,2022-03-25 00:00:00,Não Detectado,Não Detectado,Detectado,Não Detectado,Swab nasofaringe,NaN,...,20220405_Painel Viral 4.xlsx,NaN,NaN,NaN,RT-qPCR,MJCDM,2022-03-05 00:00:00,NaN,NaN,NaN
8,0,Feminino,2199885,2022-03-25 00:00:00,Não Detectado,Não Detectado,Detectado,Não Detectado,Aspirado traqueal,NaN,...,20220329_Painel Viral 4_Segunda quinzena de Ma...,NaN,NaN,NaN,RT-qPCR,AAP,2022-01-21 00:00:00,GOIANIA,NaN,NaN
3,0,Feminino,2199885,2022-03-25 00:00:00,Não Detectado,Não Detectado,Detectado,Não Detectado,Aspirado traqueal,NaN,...,20220405_Painel Viral 4.xlsx,NaN,NaN,NaN,RT-qPCR,AAP,2022-01-21 00:00:00,NaN,NaN,NaN
9,0,Feminino,2199886,2022-02-25 00:00:00,Não Detectado,Não Detectado,Detectado,Não Detectado,Aspirado traqueal,NaN,...,20220329_Painel Viral 4_Segunda quinzena de Ma...,NaN,NaN,NaN,RT-qPCR,MAFP,2022-02-22 00:00:00,GOIANIA,NaN,NaN
4,0,Feminino,2199886,2022-02-25 00:00:00,Não Detectado,Não Detectado,Detectado,Não Detectado,Aspirado traqueal,NaN,...,20220405_Painel Viral 4.xlsx,NaN,NaN,NaN,RT-qPCR,MAFP,2022-02-22 00:00:00,NaN,NaN,NaN
10,1,Masculino,2200182,2022-03-24 00:00:00,Não Detectado,Não Detectado,Detectado,Não Detectado,Swab nasofaringe,NaN,...,20220329_Painel Viral 4_Segunda quinzena de Ma...,NaN,NaN,NaN,RT-qPCR,BRGP,2021-02-21 00:00:00,GOIANIA,NaN,NaN
5,1,Masculino,2200182,2022-03-24 00:00:00,Não Detectado,Não Detectado,Detectado,Não Detectado,Swab nasofaringe,NaN,...,20220405_Painel Viral 4.xlsx,NaN,NaN,NaN,RT-qPCR,BRGP,2021-02-21 00:00:00,NaN,NaN,NaN


In [56]:
df_hla_covid['Idade'].unique()

array(['7', '85', '2', '1', '0', '38', '9', '6', '52', '88', '39', '41',
       '44', '81', '11', '3', '21', '73', '84', '5', '100', '67', '71',
       '4', '26', '75', '79', '33', '37', '27', '66', '43', '56', '90',
       '62', '34', '55', '61', '78', '36', '8', '40', '57', '74', '69',
       '65', '14', '10', '16', '31', '15', '29', '32', '64', '49', '42',
       '17', '76', '72', '77', '82', '13', '12', '60', '23', '35', '46',
       '83', '', '24', '86', '50', '91', '63', '54', '30', '51', '87',
       '58', '80', '45', '59', '25', '96', '53', '92', '70', '19', '28',
       '18', '89', '68', '22', '47', '48', '20', '98', '93', '94'],
      dtype=object)

In [57]:
df_hla_covid.query("Idade == ''")

,Idade,Sexo,Pedido,Data Coleta,Vírus Influenza A,Vírus Influenza B,Vírus Sincicial Respiratório A/B,Coronavírus SARS-CoV-2,Tipo Material,Dt Liberação,...,filename,Metodo,Métodologia,Metodologia,Metodologia Do Teste,Nome Paciente,Dt. Nascimento,Cidade.1,Matodologia,Inst Saúde
8,,Feminino,2342519,2023-03-19 00:00:00,Não Detectado,Não Detectado,Não Detectado,Não Detectado,Aspirado traqueal,2023-03-19 00:00:00,...,20230329_PR4_SE12.xlsx,NaN,qPCR,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [58]:
df_hla_covid['Sexo'].unique()

array(['Masculino', 'Feminino', 'MN'], dtype=object)

In [59]:
df_hla_covid['Data Coleta'].replace(re.compile('[0-9]'), 'x').unique()

array(['xxxx-xx-xx xx:xx:xx'], dtype=object)

In [61]:
virus_columns = [ col for col in df_hla_covid.columns if col.startswith('Vírus') ]
unique_values = set()
for col in virus_columns:
    unique_values = unique_values.union(set(df_hla_covid[col].unique()))

unique_values

{'Detectado', 'Não Detectado'}

- Drop duplicated Pedido
- Filter columns

        ``
        COVID_COLUMNS = [
            'Pedido',

            'Idade', 'Sexo',  
            
            'Cidade', 'UF',

            'Data Coleta', 'Dt Liberação',


            'Vírus Influenza A',
            'Vírus Influenza B', 
            'Vírus Sincicial Respiratório A/B',
            'Coronavírus SARS-CoV-2', 
        ]
        ``

- Fix idade -> int e remover >150
- Fix sexo -> uppercase -> F ou M se Masculino ou Feminino e outros -> ''
- Fix Data Coleta -> 'xxxx-xx-xx xx:xx:xx' -> 'yyyy-mm-dd'
- Cidade, UF ok
- Fix VIRUS_* -> Detectado, Não Detectado -> Neg, Pos
- test_kit -> test_4


### CT_N

In [62]:
df_hla_ctn.info()

<class 'pandas.core.frame.DataFrame'>
Index: 51722 entries, 0 to 771
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Idade                  51722 non-null  object
 1   Sexo                   51722 non-null  object
 2   Pedido                 51722 non-null  object
 3   Data Coleta            51722 non-null  object
 4   CT_I                   51722 non-null  object
 5   CT_N                   51722 non-null  object
 6   CT_ORF1AB              51722 non-null  object
 7   Método                 37626 non-null  object
 8   Cidade                 51722 non-null  object
 9   UF                     51078 non-null  object
 10  Resultado              51722 non-null  object
 11  filename               51722 non-null  object
 12  Semana Epidemiológica  26426 non-null  object
 13  Unidade de coleta      4219 non-null   object
 14  Métodologia            2579 non-null   object
 15  Metodologia            681

In [65]:
df_hla_ctn.columns

ctn_columns = [
    "Pedido",
    "Idade",
    "Dt. Nascimento",
    "Sexo",
    "Data Coleta",
    
    # "CT_I",

    "CT_N",
    "CT_ORF1AB",
    "Cidade",
    "UF",
    "Resultado",
]

In [66]:
df_hla_ctn[ctn_columns].head()

,Pedido,Idade,Dt. Nascimento,Sexo,Data Coleta,CT_I,CT_N,CT_ORF1AB,Cidade,UF,Resultado
0,2329024,33,NaN,Masculino,2023-01-21 00:00:00,"27,3728523254395",,,GOIANIA,GO,Não detectado (Ausência do RNA de Coronavírus ...
1,2329025,75,NaN,Feminino,2023-01-21 00:00:00,"33,5404396057129",,,GOIANIA,GO,Não detectado (Ausência do RNA de Coronavírus ...
2,2329026,82,NaN,Masculino,2023-01-21 00:00:00,"31,4492721557617",,,GOIANIA,GO,Não detectado (Ausência do RNA de Coronavírus ...
3,2329027,51,NaN,Masculino,2023-01-22 00:00:00,"30,0627174377441",,,GOIANIA,GO,Não detectado (Ausência do RNA de Coronavírus ...
4,2329028,79,NaN,Masculino,2023-01-21 00:00:00,"26,108304977417",,,GOIANIA,GO,Não detectado (Ausência do RNA de Coronavírus ...


In [67]:
df_hla_ctn['Idade'].unique()

array(['33', '75', '82', '51', '79', '67', '55', '44', '18', '29', '22',
       '68', '26', '40', '25', '80', '42', '59', '34', '57', '32', '3',
       '49', '7', '43', '45', '1', '73', '11', '58', '65', '53', '27',
       '46', '39', '52', '31', '70', '28', '86', '74', '12', '38', '89',
       '77', '0', '20', '62', '91', '81', '66', '72', '41', '36', '56',
       '76', '54', '61', '63', '60', '83', '90', '35', '47', '8', '64',
       '24', '5', '50', '17', '87', '99', '30', '71', '23', '2', '37',
       '85', '78', '16', '19', '15', '21', '69', '14', '48', '6', '10',
       '84', '4', '88', '9', '97', '13', '', '94', '93', '92', '104',
       '96', '95', '101', '105', '98', '112', '100', '107', '111', '102',
       '106', '113', '65535', '103'], dtype=object)

In [68]:
df_hla_ctn.query("Idade == '65535'")

,Idade,Sexo,Pedido,Data Coleta,CT_I,CT_N,CT_ORF1AB,Método,Cidade,UF,...,Unidade de coleta,Métodologia,Metodologia,Inst Saúde,Métodod,Métodogia,Mãe,metodologia,Metodo,Dt. Nascimento
2,65535,Masculino,2359552,2023-05-27 00:00:00,"27,9659976959229",,,RT-qPCR,APARECIDA DE GOIANIA,GO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [70]:
df_hla_ctn['Sexo'].unique()

array(['Masculino', 'Feminino', '', 'FEMININO'], dtype=object)

In [71]:
df_hla_ctn['Data Coleta'].replace(re.compile('[0-9]'), 'x').unique()

array(['xxxx-xx-xx xx:xx:xx'], dtype=object)

In [72]:
df_hla_ctn['Resultado'].unique()

array(['Não detectado (Ausência do RNA de Coronavírus SARS',
       'Detectado (Presença do RNA de Coronavírus SARS-CoV',
       'Inconclusivo'], dtype=object)

In [77]:
df_hla_ctn['Idade'].where(df_hla_ctn['Idade'] != '', '-1').astype(int)

0      33
1      75
2      82
3      51
4      79
       ..
767     9
768    69
769    63
770    76
771    23
Name: Idade, Length: 51722, dtype: int64

In [81]:
df_hla_ctn['Sexo'].str.upper().map({"FEMININO": "F", "MASCULINO": "M"}).fillna("I")

0      M
1      F
2      M
3      M
4      M
      ..
767    F
768    F
769    F
770    M
771    F
Name: Sexo, Length: 51722, dtype: object

- Drop duplicated Pedido
- Filter columns

        ``
        COVID_COLUMNS = [
            'Pedido',

            'Idade', 'Sexo',  
            
            'Cidade', 'UF',

            'Data Coleta', 'Dt Liberação',


            'Vírus Influenza A',
            'Vírus Influenza B', 
            'Vírus Sincicial Respiratório A/B',
            'Coronavírus SARS-CoV-2', 
        ]
        ``

- Fix idade -> int e remover >150
- Fix sexo -> uppercase -> F ou M se Masculino ou Feminino e outros -> ''
- Fix CT_N	CT_ORF1AB	-> Substituir , por . e transformar
- Fix Data Coleta -> 'xxxx-xx-xx xx:xx:xx' -> 'yyyy-mm-dd'
- Cidade, UF ok
- Fix Resultado -> Startswith Detectado, Não Detectado, Inconclusivo -> Neg, Pos
- test_kit -> covid_pcr